In [1]:
from bs4 import BeautifulSoup
import pandas as pd
import os

In [2]:
def extract_LRPs(filepath):
    with open(filepath, 'r', encoding='ISO-8859-1') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Find table by looking for rows with class 'tdRow' (data rows)
    lrp_table = None
    for table in soup.find_all('table'):
        if table.find('td', class_='tdRow'):
            lrp_table = table
            break

    rows = lrp_table.find_all('tr')

    records = []
    for row in rows:
        cells = row.find_all('td', class_='tdRow')
        if len(cells) < 7:
            continue

        records.append({
            'LRP_No':            cells[1].get_text(strip=True),
            'Road_Chainage':     cells[2].get_text(strip=True),
            'LRP_Type':          cells[3].get_text(strip=True),
            'Description':       cells[4].get_text(strip=True),
            'Latitude_Decimal':  cells[5].get_text(strip=True),
            'Longitude_Decimal': cells[6].get_text(strip=True),
        })

    return pd.DataFrame(records)

In [3]:
def iterate_scrape_LRPs(folder, file_list):
    all_dfs = []

    for filename in os.listdir(folder):
        # 1. Check extension first
        if filename.endswith('.lrps.htm'):
            # 2. Extract the road name (e.g., 'N1')
            road_name = filename.replace('.lrps.htm', '')

            # 3. Check if that exact road name is in your target list
            if road_name in file_list:
                filepath = os.path.join(folder, filename)
                df = extract_LRPs(filepath)
                df['Road'] = road_name
                all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
# grabs all N-roads e.g. N1, N2...

In [4]:
folder = r'../data/rmms'

roads = ['N1', 'N2', 'N102', 'N104', 'N105', 'N106', 'N204', 'N207', 'N208']
df_all_lrp = iterate_scrape_LRPs(folder, roads)
# df_all_lrp.to_csv(r'../data/roads_lrp.csv', index=False)

In [5]:
df_all_lrp.head()

,LRP_No,Road_Chainage,LRP_Type,Description,Latitude_Decimal,Longitude_Decimal,Road
0,LRPS,0,Others,Road Start from N207 infront of RHD,24.4713604,91.7655556,N208
1,LRPS,0,Others,Road Start from N207 infront of RHD,24.4713604,91.7655556,N208
2,LRPS,0,Others,Road Start from N207 infront of RHD,24.4713604,91.7655556,N208
3,LRP001,1,Km Post,Km post missing,24.4757216,91.7727771,N208
4,LRP001a,1.305,Culvert,Box culvert,24.4756382,91.7758049,N208


In [6]:
def extract_traffic(filepath):
    with open(filepath, 'r', encoding='ISO-8859-1') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Find the table with data rows
    lrp_table = None
    for table in soup.find_all('table'):
        if table.find('td', class_='tdRow'):
            lrp_table = table
            break

    # Manually define column names since the header spans multiple rows with colspans
    columns = [
        'Link_No', 'Name',
        'Start_LRP', 'Start_Offset', 'Start_Chainage',
        'End_LRP', 'End_Offset', 'End_Chainage',
        'Length_km',
        'Heavy_Truck', 'Medium_Truck', 'Small_Truck',
        'Large_Bus', 'Medium_Bus', 'Micro_Bus',
        'Utility', 'Car', 'Auto_Rickshaw', 'Motor_Cycle',
        'Bi_Cycle', 'Cycle_Rickshaw', 'Cart',
        'Motorized', 'Non_Motorized', 'Total_AADT', 'AADT'
    ]

    records = []
    for row in lrp_table.find_all('tr'):
        cells = row.find_all('td', class_='tdRow')
        if len(cells) < len(columns):
            continue  # skip header rows

        records.append({
            col: cells[i].get_text(strip=True)
            for i, col in enumerate(columns)
        })

    return pd.DataFrame(records)

N1, N2, and roads longer than 25km (found in assignment3): N102, N104, N105, N106,
N204, N207, and N208.

In [7]:
def iterate_scrape_traffic(folder, road_names):
    all_dfs = []

    for road in road_names:
        filename = f'{road}.traffic.htm'  # adjust if the actual filename pattern is different
        filepath = os.path.join(folder, filename)
        df = extract_traffic(filepath)
        df['Road'] = road
        all_dfs.append(df)
    return pd.concat(all_dfs, ignore_index=True)

folder = r'../data/rmms'

roads = ['N1', 'N2', 'N102', 'N104', 'N105', 'N106', 'N204', 'N207', 'N208']
df_all_traffic = iterate_scrape_traffic(folder, roads)
# df_all_traffic.to_csv(r'../data/roads_traffic.csv', index=False)



In [8]:
df_all_traffic.head()

,Link_No,Name,Start_LRP,Start_Offset,Start_Chainage,End_LRP,End_Offset,End_Chainage,Length_km,Heavy_Truck,...,Auto_Rickshaw,Motor_Cycle,Bi_Cycle,Cycle_Rickshaw,Cart,Motorized,Non_Motorized,Total_AADT,AADT,Road
0,N1-1L,Jatrabari - Int.with Z1101 (Left) (Left),LRPS,0,0,LRPS,822,0.822,0.822,402.0,...,2980.0,398.0,232.0,889.0,0.0,18236.0,1121.0,19357.0,19357.0,N1
1,N1-1L,Jatrabari - Int.with Z1101 (Left) (Left),LRPS,0,0,LRPS,822,0.822,0.822,402.0,...,2980.0,398.0,232.0,889.0,0.0,18236.0,1121.0,19357.0,19357.0,N1
2,N1-1L,Jatrabari - Int.with Z1101 (Left) (Left),LRPS,0,0,LRPS,822,0.822,0.822,402.0,...,2980.0,398.0,232.0,889.0,0.0,18236.0,1121.0,19357.0,19357.0,N1
3,N1-1R,Jatrabari - Int.with Z1101 (Left) (Right),LRPS,0,0,LRPS,822,0.822,0.822,660.0,...,2508.0,436.0,213.0,1088.0,0.0,20236.0,1301.0,21537.0,21537.0,N1
4,N1-2L,Int.with Z1101 - Signboard (Left) R111 (Left),LRPS,822,0.822,LRPS,4175,4.175,3.353,660.0,...,2508.0,436.0,213.0,1088.0,0.0,20236.0,1301.0,21537.0,21537.0,N1


In [9]:
network_df = pd.read_csv("../data/network_data.csv")

In [10]:
def process_lrp_traffic_data(network_data, traffic_data):
    # Ensure we only have the columns we need to avoid cluttering the merge
    network_data.columns = network_data.columns.str.lower()
    traffic_data.columns = traffic_data.columns.str.lower()
    traffic_data = traffic_data.rename(columns={'start_lrp': 'lrp'})
    traffic_cols = ['road', 'lrp', 'small_truck', 'medium_truck', 'heavy_truck', 'aadt']

    traffic_subset = traffic_data[traffic_cols].copy()
    traffic_subset = traffic_subset.drop_duplicates(subset=['road', 'lrp'], keep='first')

    #First try based on matching road and lrp
    # We merge on 'road' and 'lrp'.
    # Note: ensure column names match (e.g., 'road' vs 'Road')
    integrated_df = pd.merge(
        network_data,
        traffic_subset,
        on=['road', 'lrp'],
        how='left'
    )

    #if don't have use forward fill
    # Sort by road and lrp to ensure 'forward fill' picks the segment 'above' (ffill)
    integrated_df = integrated_df.sort_values(by=['road', 'id'], ascending=[True, True])
    integrated_df = integrated_df.sort_values(['road', 'lrp'])

    truck_cols = ['small_truck', 'medium_truck', 'heavy_truck', 'aadt']
    integrated_df[truck_cols] = integrated_df[truck_cols].apply(pd.to_numeric, errors='coerce')
    integrated_df[truck_cols] = integrated_df.groupby('road')[truck_cols].ffill()

    # if also don't have then take the mean for that road

    road_means = integrated_df.groupby('road')[truck_cols].transform('mean')
    integrated_df[truck_cols] = integrated_df[truck_cols].fillna(road_means)

    return integrated_df


integrated_data = process_lrp_traffic_data(network_df, df_all_traffic)
integrated_data.head()



,road,id,model_type,name,lat,lon,length,condition,lrp,small_truck,medium_truck,heavy_truck,aadt
3,N1,1000003,link,link 3,23.702139,90.451972,168,NaN,LRP001,692.02243,2381.054206,179.993769,10036.032399
4,N1,1000004,link,link 4,23.697889,90.460583,996,NaN,LRP002,692.02243,2381.054206,179.993769,10036.032399
5,N1,1000005,link,link 5,23.697361,90.461667,125,NaN,LRP002a,692.02243,2381.054206,179.993769,10036.032399
6,N1,1000006,link,link 6,23.693833,90.469138,856,NaN,LRP003,692.02243,2381.054206,179.993769,10036.032399
7,N1,1000007,link,link 7,23.693611,90.478777,982,NaN,LRP004,692.02243,2381.054206,179.993769,10036.032399


In [11]:
# Verify row count matches
print(f"Integrated data: {len(integrated_data)} rows")


Integrated data: 4083 rows


In [12]:
# Reorder rows to match original network_data.csv sequence —
# the model relies on components being in sequential road order
network_order = pd.read_csv("../data/network_data.csv")[["road", "id"]]
integrated_data = network_order.merge(integrated_data, on=["road", "id"], how="left")
integrated_data.to_csv(r"../data/integrated_data.csv", index=False)
print(f"Saved {len(integrated_data)} rows to integrated_data.csv")


Saved 4083 rows to integrated_data.csv
